[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/modal_xpbd/modal_xpbd.ipynb)

# Flexible Bodies, Rigidly Joined

Girders of stiff bars, pinned together at their corners. Each girder's large motion is a motor, and its small bending is a few of its vibration modes: fields of displacements over its points, each with an amplitude. Two scenes: a chain of girders hinged at one corner swings and folds under gravity, and a beam of girders spliced end to end, clamped at both ends, is crushed until it buckles. The bodies are stepped by extended position-based dynamics: each step predicts the motion, then moves the bodies until every pin closes, all pins solved together. Every operator below is an extensor, and the couplings between bodies and pins are sparse extensors.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from dataclasses import replace

import numpy as np
from IPython.display import Image, display

from numga import stack
from numga.sparse import SparseExtensor, spdiag
from examples.animation import save_animation
from examples.mechanics import lie_integrators as lie
from examples.mechanics.modal_xpbd import core, render
from examples.mechanics.modal_xpbd.core import (
    Bodies, Direction, Force, Forque, Motor, Pins, Point, Scalar, Shape, Twist, incidence, mv, truss,
)
from examples.mechanics.modal_xpbd.scenarios import clamped_beam, critical, crushed, hinged_chain

# The chain's girder: its cells, length and height, its bars' stiffness, its density, and the
# vibration modes it keeps.
CELLS, LENGTH, HEIGHT = 4, 1.0, 0.2
STIFFNESS, DENSITY, MODES = 300.0, 1.0, 8
# The chain's moving girders, their modes' damping ratio, and gravity.
LINKS = 4
DAMPING = np.array([0.02])
GRAVITY = (mv.y * -4).dual()                                               # [] Direction
# The swing's time step, its frames and the steps between them, and the frame duration.
INTERVAL, FRAMES, SUBSTEPS = 0.002, 200, 15
DURATION_MS = 30
# The modes drawn, and how far each is drawn displaced.
SHOWN, DRAWN = 4, 0.03
# The beam: its moving girders, each of unit cells as long as it has cells, as stiff as steel wire
# against its weight, overdamped; the end displacement it is crushed by, and its frames.
BEAM_GIRDERS, BEAM_CELLS, BEAM_HEIGHT = 8, 32, 1.0
BEAM_STIFFNESS, BEAM_DAMPING = 1e11, np.array([3.0])
CRUSH, BEAM_FRAMES, BEAM_SUBSTEPS, BEAM_INTERVAL = 0.15, 120, 20, 0.002
WEIGHTLESS = (mv.y * 0).dual()                                             # [] Direction

## 1. The girder and its vibrations

The girder is a grid of bars between points, laid out by `truss`. Displacements and forces in it are the lines `x` and `y` through the origin, `Force`, the polars of the directions: joined with a direction, `force & direction` is the work done moving that way. `ends` is a sparse extensor from fields on the points to fields on the bars: each bar couples to its two end points through the scalars minus one and one, so `ends * positions` is every bar's difference, tail from head. Each bar is a map `Force <- Force`, `direction * (direction | Force)`, that measures a displacement along itself and pulls back along the same line. Map cells compose by call, so `~ends * bars(ends * Force)` is the girder's stiffness: the bars' stretches, their pulls, and the reverse carrying the pulls back to the points. Its eigenfields against the masses are the vibration modes; the three at zero are the rigid motions, which belong to the motor. Below are the four slowest.

In [ ]:
positions, edges = truss(CELLS, LENGTH, HEIGHT)                         # [vertices] Force, [edges, ends]
vertices = positions.shape[0]
# Each bar's tail taken from its head.
ends = SparseExtensor.from_columns(edges, mv.scalar((np.ones_like(edges) * [-1, 1])[..., None]), vertices)   # [edges, vertices] Scalar
difference = ends * positions                                              # [edges] Force
lengths = difference.norm()                                                # [edges] Scalar
directions = difference / lengths                                          # [edges] Force
# Each bar measures extension along itself and pulls back along the same direction.
bars = spdiag(directions * (directions | Force) * STIFFNESS / lengths)       # [edges, edges] Force <- Force
masses = np.full(vertices, DENSITY * LENGTH * HEIGHT / vertices)
weights = spdiag(mv.scalar(masses[:, None])) * Force                         # [vertices, vertices] Force <- Force
# The stiffness against the masses; its three motions at zero are rigid and belong to the motor.
values, fields = (~ends * bars(ends * Force)).eigh(weights, 3 + MODES)     # [3 + modes] Scalar, [3 + modes, vertices] Force
values, fields = values[3:], fields[3:]                                    # [modes] Scalar, [modes, vertices] Force
centre = (positions * masses).sum(axis=0) / masses.sum()                   # [] Force
rest = (positions - centre + mv.w).dual()                                  # [vertices] Point
inertia = ((rest & rest.commutator(Twist)) * masses).sum(axis=0)           # [] Forque <- Twist
shape = Shape(rest, fields.dual(), values.square_root(), 1 / values, masses, edges, inertia)

print(ends.gatype, ends.shape)
print(bars.gatype, bars.shape)
render.mode_shapes(shape.rest, shape.rest + shape.modes[:SHOWN] * DRAWN, shape.edges);

## 2. Bodies and pins

A point of a girder is its rest point plus the modes weighted by their amplitudes, carried into the world by the body's motor, `motor >> local`. The bodies are batched: one field per quantity, with an axis over the bodies. A pin joins an anchor point on one body to one on another: a hinge is one pin, at the upper corners where two girders meet, and a splice two, at both corners. The chain's first girder is fixed, its inverse inertia zero.

In [ ]:
def points(bodies: Bodies, shape: Shape) -> Point:
    """Every body's points in the world: its rest points moved by its modes, carried by its motor."""
    local = shape.rest + (shape.modes * bodies.amplitudes[..., None]).sum(axis=-2)   # [..., bodies, vertices] Point
    return bodies.motor[..., None] >> local                                # [..., bodies, vertices] Point


bodies, pins = hinged_chain(shape, LINKS, DAMPING)
render.draw_chain(points(bodies, shape), shape.edges);

## 3. How the pins open

How an anchor moves for a twist of its body is a map. With the twist left open, the commutator `anchor.commutator(Twist)` is that map, `Direction <- Twist`, and the motor carries it into the world; a pin's two anchors, taken from each other, say how the pin opens. Over all pins and bodies these maps are the cells of a sparse extensor, `motion`, from the bodies' twists to how every pin opens, `[pins, bodies] Direction <- Twist`, coupling each pin to its two bodies only. Its adjugate goes the other way: the forces at the pins as the forques they exert on the bodies, with `motion.adjugate()(forces) & twists == forces & motion(twists)` summed over the elements, the same work either way. The modes do the same through `shapes`, from the bodies' mode amplitudes to how the pins open, and `loads`, the work a force at a pin does through each mode.

In [ ]:
def coupling(bodies: Bodies, pins: Pins) -> tuple[SparseExtensor, SparseExtensor, SparseExtensor, Direction]:
    """How every pin opens for the bodies' twists and for their modes' amplitudes, the work a force at each
    pin does through each mode, and the gap."""
    count, modes = bodies.motor.shape[-1], bodies.amplitudes.shape[-1]
    motor = bodies.motor[..., pins.bodies]                                 # [..., pins, ends] Motor
    local = pins.anchors + (pins.modes * bodies.amplitudes[..., pins.bodies, :]).sum(axis=-1)   # [..., pins, ends] Point
    signs = np.array([-1, 1])
    # An open twist measures how an anchor moves, the two ends taken from each other.
    motion = (motor >> local.commutator(Twist)) * signs                    # [..., pins, ends] Direction <- Twist
    shapes = (motor[..., None] >> pins.modes) * signs[:, None]             # [..., pins, ends, modes] Direction
    gap = ((motor >> local) * signs).sum(axis=-1)                          # [..., pins] Direction
    pin = np.broadcast_to(np.arange(len(pins.bodies))[:, None], pins.bodies.shape)   # [pins, ends]
    mode = pins.bodies[..., None] * modes + np.arange(modes)               # [pins, ends, modes]
    pin_mode = np.broadcast_to(pin[..., None], mode.shape)                 # [pins, ends, modes]
    return (
        incidence(motion, pin, pins.bodies, (len(pins.bodies), count)),    # [..., pins, bodies] Direction <- Twist
        incidence(shapes, pin_mode, mode, (len(pins.bodies), count * modes)),   # [..., pins, bodies * modes] Direction
        incidence(Force & shapes, mode, pin_mode, (count * modes, len(pins.bodies))),   # [..., bodies * modes, pins] Scalar <- Force
        gap,
    )


motion, shapes, loads, gap = coupling(bodies, pins)
print(motion.gatype, motion.shape)
print(loads.gatype, loads.shape)

## 4. Closing every pin together

Composed through the inverse inertia, `motion(inertias(motion.adjugate()))` is how every pin opens for a force at any other, `[pins, pins] Direction <- Force`, with no Jacobian matrix assembled. Each mode is a spring with an implicit dashpot, its compliance scaled by the step; over a step it yields to a unit load by `compliance / (1 + compliance)`, its mobility, and adds `shapes * (mobility * loads)`. Each pin's own compliance adds the polar `Force.dual()`, which moves its anchors apart along the force. One sparse solve of the sum gives the forces that close every pin together. Solved one joint at a time instead, a stiff structure converges too slowly to carry a load: the joints give way before the girders bend.

In the notation of extended position-based dynamics the system reads as $S = J_b M_b^{-1} J_b^\top + J_q W J_q^\top + \tilde\alpha_p$, with $W = \tilde\alpha_m / (1 + \tilde\alpha_m)$ the modes' mobility.

In [ ]:
def modal_terms(bodies: Bodies, previous: Scalar, dt: float) -> tuple[Scalar, Scalar, Scalar]:
    """Each mode's compliance and residual over the step, with its implicit dashpot folded in, and how
    far it yields to a unit load."""
    damping = 2 * bodies.damping * bodies.frequencies * bodies.compliance / dt   # [..., bodies, modes] Scalar
    compliance = bodies.compliance / dt**2 / (1 + damping)                 # [..., bodies, modes] Scalar
    residual = (bodies.amplitudes + damping * (bodies.amplitudes - previous)) / (1 + damping)   # [..., bodies, modes] Scalar
    return compliance, residual, 1 / (1 + compliance)


def project(bodies: Bodies, pins: Pins, previous: Scalar, lambdas: Scalar, dt: float) -> tuple[Bodies, Scalar]:
    """The bodies moved so that every pin closes, all solved together, and the modes' accumulated
    reactions."""
    compliance, residual, yielding = modal_terms(bodies, previous, dt)      # [..., bodies, modes] Scalar each
    motion, shapes, loads, gap = coupling(bodies, pins)
    batch = bodies.amplitudes.shape[:-2]
    mode_load = -(residual + compliance * lambdas)                         # [..., bodies, modes] Scalar
    free_step = (yielding * mode_load).reshape(batch + (-1,))              # [..., bodies * modes] Scalar
    mobility = spdiag((yielding * compliance).reshape(batch + (-1,)))         # [..., bodies * modes, bodies * modes] Scalar
    weld = spdiag(pins.compliance / dt**2 * Force.dual())                  # [pins, pins] Direction <- Force
    inertias = spdiag(bodies.inverse_inertia)                                # [..., bodies, bodies] Twist <- Forque
    system = motion(inertias(motion.adjugate())) + shapes * (mobility * loads) + weld   # [..., pins, pins] Direction <- Force
    forces = system.solve((-gap - shapes * free_step).cast(Direction))     # [..., pins] Force
    # Equal and opposite forces act through both the rigid and elastic responses.
    displacement = bodies.inverse_inertia(motion.adjugate()(forces))       # [..., bodies] Twist
    mode_forces = loads(forces).reshape(bodies.amplitudes.shape)           # [..., bodies, modes] Scalar
    modal_reaction = yielding * (mode_load - mode_forces)                   # [..., bodies, modes] Scalar
    moved = replace(bodies, motor=bodies.motor * (displacement * -0.5).exp(),
                    amplitudes=bodies.amplitudes + mode_forces + modal_reaction)
    return moved, lambdas + modal_reaction

## 5. A step

A step predicts each body's motion from the rigid-body equations under its weight, `lie.explicit_rk4`, and moves the modes by their rates. The springs relax, every pin closes, and the new rates are read from how far everything moved, the motors' by the logarithm of their change.

In [ ]:
def step(bodies: Bodies, pins: Pins, dt: float, gravity: Direction) -> Bodies:
    """The bodies one step on."""
    previous = bodies

    def weight(motor: Motor, rate: Twist) -> Forque:
        # Gravity pulls through each body's centre of mass, the origin of its frame.
        return (mv.w.dual() & (motor << gravity)) * bodies.masses           # [..., bodies] Forque

    motor, _ = lie.explicit_rk4(bodies.motor, bodies.rate, bodies.inertia, bodies.inverse_inertia, dt, weight)
    bodies = replace(bodies, motor=motor, amplitudes=bodies.amplitudes + bodies.rates * dt)
    _, residual, yielding = modal_terms(bodies, previous.amplitudes, dt)   # [..., bodies, modes] Scalar each
    lambdas = -residual * yielding                                          # [..., bodies, modes] Scalar
    bodies, _ = project(replace(bodies, amplitudes=bodies.amplitudes + lambdas), pins, previous.amplitudes, lambdas, dt)
    return replace(bodies, rate=(~previous.motor * bodies.motor).log() * (-2 / dt),
                   rates=(bodies.amplitudes - previous.amplitudes) / dt)

## 6. The swing

Released from horizontal, the chain swings and folds. The large turns belong to the motors, and the bending within each girder stays small; damping acts only on the vibrations. Inertia is evaluated at rest and the coupling of rotation and vibration is left out, so the girders should turn slowly compared with their vibrations, and they pass through each other: the example models the pins and the elastic response, not contact.

In [ ]:
def swing(bodies: Bodies):
    """The chain's points at every frame."""
    for _ in range(FRAMES):
        yield points(bodies, shape)                                        # [cases, bodies, vertices] Point
        for _ in range(SUBSTEPS):
            bodies = step(bodies, pins, INTERVAL, GRAVITY)


display(Image(filename=save_animation(render.swinging_chain(swing(bodies), shape.edges, points(bodies, shape)), "modal_xpbd_swing", DURATION_MS)))

## 7. The buckling beam

Eight girders as stiff as steel wire, spliced end to end and clamped at both ends, have one clamp driven inward a little further every frame. There is no imperfection: the beam stays straight, compressed, until the end displacement crosses its Euler load, and then pops sideways, the symmetry broken by rounding alone. A single linearized girder cannot buckle, as its constant stiffness has linearized away how an axial load couples into bending; the chain recovers it, as the large rotations live in the moving frames. The pop trails the Euler estimate a little: past it the straight beam is still an equilibrium, only an unstable one, and rounding noise takes a while to grow.

For two chords a height $h$ apart and a span $L$, the Euler load of a beam clamped at both ends, $4\pi^2 EI / L^2$ with $EI = EA\,h^2/2$, over its axial stiffness $2EA/L$, is an end displacement of $\pi^2 h^2 / L$.

In [ ]:
beam = core.girder(BEAM_CELLS, float(BEAM_CELLS), BEAM_HEIGHT, BEAM_STIFFNESS, DENSITY, MODES)
clamped, splices = clamped_beam(beam, BEAM_GIRDERS, BEAM_DAMPING)
crushing = CRUSH * np.arange(BEAM_FRAMES) / BEAM_FRAMES                     # [frames]


def crush(bodies: Bodies):
    """The beam's points and its midspan, the middle of its two middle girders' centres, at every frame."""
    rest, middle = bodies.motor, bodies.motor.shape[-1] // 2
    for displacement in crushing:
        bodies = crushed(bodies, rest, displacement)
        for _ in range(BEAM_SUBSTEPS):
            bodies = step(bodies, splices, BEAM_INTERVAL, WEIGHTLESS)
        centres = bodies.motor[..., middle - 1:middle + 1] >> mv.w.dual()  # [cases, 2] Point
        yield points(bodies, beam), centres.sum(axis=-1) / 2                 # [cases, bodies, vertices] Point, [cases] Point


frames, midspans = zip(*crush(clamped))
midspans = stack(midspans)                                                  # [frames, cases] Point
euler = critical(BEAM_HEIGHT, BEAM_GIRDERS * BEAM_CELLS)
display(Image(filename=save_animation(render.beam(list(frames), beam.edges), "modal_xpbd_buckle", DURATION_MS)))
render.buckling(crushing, midspans, euler);

In [ ]:
# checks
# The girder and the step laid out here are the core's, the girder to the eigensolver's convergence.
np.testing.assert_allclose(core.girder(CELLS, LENGTH, HEIGHT, STIFFNESS, DENSITY, MODES).frequencies.kernel, shape.frequencies.kernel, rtol=1e-6)
moved, expected = bodies, bodies
for _ in range(SUBSTEPS):
    moved, expected = step(moved, pins, INTERVAL, GRAVITY), core.step(expected, pins, INTERVAL, GRAVITY)
np.testing.assert_allclose((points(moved, shape) - core.points(expected, shape)).kernel, 0.0, atol=1e-12)
# The adjugate carries forces back with the same work, summed over the elements:
# motion.adjugate()(forces) & twists == forces & motion(twists).
motion, _, _, gap = coupling(moved, pins)
forces = mv.vector(np.random.default_rng(0).normal(size=(1, len(pins.bodies), 3))).cast(Force)   # [cases, pins] Force
twists = mv.bivector(np.random.default_rng(1).normal(size=(1, LINKS + 1, 3)))                    # [cases, bodies] Twist
np.testing.assert_allclose((motion.adjugate()(forces) & twists).sum(axis=-1).kernel, (forces & motion(twists)).sum(axis=-1).kernel, rtol=1e-12)
# The hinges stay closed to well under a millimetre.
assert np.abs(gap.kernel).max() < 1e-4
# The beam stays straight below its Euler load and has popped by twice it.
deflection = (midspans[:, 0].dual() | mv.y).abs()                         # [frames] Scalar
assert deflection[crushing < 0.8 * euler].kernel.max() < 1e-6
assert deflection[crushing > 2 * euler].kernel.min() > 0.5